In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

from ma.utils import load_participant_data, put_metadata_to_segments
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import pandas as pd
import numpy as np

plt.rcParams['pgf.preamble'] = r'\usepackage{tikz}'

In [2]:
from scipy.signal import find_peaks

def plot_peaks_on_integrated_signal(integrated_signal: np.ndarray, peaks: np.ndarray) -> Figure:
    _, ax = plt.subplots(figsize=(20, 9))
    ax.plot(integrated_signal, color="blue", linestyle="-", label = "integrated signal")
    ax.scatter(peaks, integrated_signal[peaks], color = "red")

    ax.grid()
    ax.legend()

def differentiate(signal:np.ndarray):
    # Differentiate:
    diff_signal = np.diff(signal)
    return diff_signal

def square(signal: np.ndarray):
    squared_signal = signal ** 2
    return squared_signal

# Moving window integration
def moving_window_integration(signal: np.ndarray, window_size: int):
    integrated_signal = np.convolve(signal, np.ones(window_size) / window_size, mode='same')
    return integrated_signal

def pan_tompkins_segmented(df_segmented: list[pd.DataFrame], height_threshold: float = 0.5):
    all_peaks = []
    for segment_no, segment in enumerate(df_segmented):
        fs = 1 / ((segment.index[1] - segment.index[0]).total_seconds())
        ecg_signal = segment[segment.columns[0]].values
        # Differentiation
        diff_signal = differentiate(ecg_signal)

        # Squaring
        squared_signal = square(diff_signal)

        # Moving window integration
        window_size = int(0.12 * fs)
        integrated_signal = moving_window_integration(squared_signal, window_size)

        # Find peaks
        distance = int(0.2 * fs)
        ht = height_threshold * max(integrated_signal)
        peaks, _ = find_peaks(integrated_signal, distance=distance, height=ht)
        all_peaks.append(peaks + (len(segment)* segment_no))

    flattened_list = [item for sublist in all_peaks for item in sublist]

    return flattened_list

# R-Peak Detection on the Reference ECG Signal with Pan-Tompkins (No Bandpass):
- `Apply Pan-Thompson to each segment individually.`

In [3]:
parti17_ecg_ref_segmented = load_participant_data(participant_number=17, signal_names=["ecg_ref"], bandpassed=True, segmented=True)
parti17_ecg_ref = load_participant_data(participant_number=17, signal_names=["ecg_ref"], bandpassed=True, segmented=False)

In [4]:
# height_threshold = 0.015 * max(ecg_signal)
height_threshold = 0.4
r_peaks = pan_tompkins_segmented(parti17_ecg_ref_segmented,height_threshold=height_threshold)

fig, ax = plt.subplots(figsize=(20, 9))
ax.plot(parti17_ecg_ref, color="blue", linestyle="-", label = "ecg_ref")
ax.scatter(parti17_ecg_ref.index[r_peaks], parti17_ecg_ref.iloc[r_peaks]["ecg_ref"].values, color = "red")

ax.grid()
fig.legend()

# R-Peak Detection on the cushion ECG Signal:
- Skip the signals with bad quality, noisy quality and the signals with motion artifacts !

In [5]:
from ma.utils import load_participant_data, put_metadata_to_segments

parti17 = load_participant_data(
    participant_number=17,
    signal_names=["ecg1"],
    bandpassed=True,
)
parti17_data, parti17_labels = load_participant_data(
    participant_number=17,
    signal_names=["ecg1"],
    bandpassed=True,
    segmented=True,
    load_labels=True
)
parti17_segmented = put_metadata_to_segments(
    segments=parti17_data, labels=parti17_labels, parti_no=17
)

In [6]:
height_threshold = 0.4
r_peaks = pan_tompkins_segmented(df_segmented=parti17_segmented, height_threshold=height_threshold)

In [7]:
fig, ax = plt.subplots(figsize=(20, 12))
ax.plot(parti17, color="blue", linestyle="-")
ax.scatter(parti17.index[r_peaks], parti17.iloc[r_peaks]["ecg1"].values, color = "red")
ax.grid()

## Plots for Thesis 

In [8]:
%load_ext autoreload
%autoreload 2
# %matplotlib qt
from ma.utils import load_all_signals, load_participant_data, put_metadata_to_segments
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from ma.utils import load_all_reference_signals
from ma.gp_utils import ecg_r_peak_detection, calculate_hr_or_rr

plt.rcParams['pgf.preamble'] = r'\usepackage{tikz}'

CLASS_NAME = "HR"
REF_SIGNAL = "ecg_ref"
SIGNAL_TYPE = "ECG"
SIGNAL = "ecg1"

# Get the discriminated data:
all_data = load_all_signals(signal_type=SIGNAL_TYPE, bandpassed=True, normalized=False, class_name=CLASS_NAME)
ref_data = load_all_reference_signals(REF_SIGNAL, CLASS_NAME)

# Get the classes
good_no_ma = []
good_ma = []
bad_no_ma = []
bad_ma = []
noisy_no_ma = []
noisy_ma = []

for segment_no, segment in enumerate(all_data):
    if segment.attrs[f"{SIGNAL}_sq"] == 0 and segment.attrs[f"{SIGNAL}_ma"] == 0:
        good_no_ma.append(segment_no)
    if segment.attrs[f"{SIGNAL}_sq"] == 0 and segment.attrs[f"{SIGNAL}_ma"] == 1:
        good_ma.append(segment_no)
    if segment.attrs[f"{SIGNAL}_sq"] == 1 and segment.attrs[f"{SIGNAL}_ma"] == 0:
        bad_no_ma.append(segment_no)
    if segment.attrs[f"{SIGNAL}_sq"] == 1 and segment.attrs[f"{SIGNAL}_ma"] == 1:
        bad_ma.append(segment_no)
    if segment.attrs[f"{SIGNAL}_sq"] == 2 and segment.attrs[f"{SIGNAL}_ma"] == 0:
        noisy_no_ma.append(segment_no)
    if segment.attrs[f"{SIGNAL}_sq"] == 2 and segment.attrs[f"{SIGNAL}_ma"] == 1:
        noisy_ma.append(segment_no)
def get_random_signals(
    signal1: tuple[str], signal2: tuple[str], signal3: tuple[str]
) -> list[pd.DataFrame]:
    conditions_map = {
        ("good", "no_ma"): good_no_ma,
        ("good", "ma"): good_ma,
        ("bad", "no_ma"): bad_no_ma,
        ("bad", "ma"): bad_ma,
        ("noisy", "no_ma"): noisy_no_ma,
        ("noisy", "ma"): noisy_ma,
    }

    def get_random_segment(condition):
        if condition in conditions_map and conditions_map[condition]:
            return random.choice(conditions_map[condition])
        return None

    random_segments = []
    for signal in [signal1, signal2, signal3]:
        segment_no = get_random_segment(signal)
        if segment_no is not None:
            random_segments.append(all_data[segment_no])

    return random_segments

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
def plot_hr_signal(df:pd.DataFrame, peaks:list, ax, label: str) -> None:
    df_copy = df.copy()
    # df_copy.index = (df_copy.index - df_copy.index[0]).total_seconds()
    df_hr = calculate_hr_or_rr(df_parti_segmented=[df_copy], peaks=[peaks], class_name="HR")
    plot_indexes = [df.index[0] + index for index in df_hr.index]
    ax.plot(plot_indexes, df_hr["hr"], marker="o", label= label)
    ax.grid()

def plot_reference_signal(df_ecg:pd.DataFrame, ax, legend:bool = False) -> None:
    parti_no = int(df_ecg.attrs["signal"].split("parti: ")[-1].split(" ")[0])
    filtered_ref_data = ref_data[ref_data["parti"] == parti_no]
    ref_start = np.argmin(abs(filtered_ref_data.index - df_ecg.index[0]))
    ref_stop = np.argmin(abs(filtered_ref_data.index - df_ecg.index[-1]))
    df_ref = filtered_ref_data.iloc[ref_start:ref_stop]

    ax2 = ax.twinx()    
    ax2.plot(            
        df_ref.index,
        df_ref[REF_SIGNAL],
        color="orange",
        alpha=0.3,
        label="Reference cECG Signal",
    )

    # Reference Signal Peaks:
    peaks_ref = ecg_r_peak_detection(df_segmented=[df_ref], signal_name=REF_SIGNAL, height_threshold=0.349)[0]
    ax2.scatter(
        df_ref.index[peaks_ref],
        df_ref.iloc[peaks_ref][REF_SIGNAL],
        linewidth=2,
        color="purple",
        alpha=1,
        label="Detected Peaks - Reference Signal",
    )
    if legend:
        ax2.legend()
    return df_ref, peaks_ref

def plot_triple_data(dfs: list[pd.DataFrame], signal: str, title: list[str]):
    fig, ax = plt.subplots(ncols=2, nrows=3, figsize=(24, 12))
    
    for df_no, df in enumerate(dfs):
        df_ecg = df[signal]
        ax[df_no][0].plot(df_ecg, label=f"Cushion Signal: {signal}")
        # Detect the peaks:
        peaks = ecg_r_peak_detection(df_segmented=[df], signal_name= "ecg1")[0]
        # Scatter the detected peaks: 
        ax[df_no][0].scatter(
            df_ecg.index[peaks],
            df_ecg.iloc[peaks],
            linewidth=2,
            color="red",
            alpha=1,
            label="Detected Peaks - Cushion Signal",
        )

        ax[df_no][0].grid()        
        ax[df_no][0].set_title(
            title[df_no]
        )
        ax[df_no][0].set_ylabel("Voltage [mV]")

        # Plot HR- Signal
        plot_hr_signal(df=df_ecg, peaks=peaks, ax=ax[df_no][1], label="HR - Cushion Signal")

        # Reference Signal:     
        df_ref, peaks_ref = plot_reference_signal(df_ecg=df_ecg, ax = ax[df_no][0])

        # Plot HR- Ref
        plot_hr_signal(df=df_ref, peaks=peaks_ref, ax= ax[df_no][1], label="HR - Reference Signal")

        ax[df_no][1].grid()    

    ax[-1][0].set_xlabel("Time [s]")
    ax[-1][1].set_xlabel("Time [s]")
    # fig.legend()
    return fig

def plot_single_data(df: pd.DataFrame, signal: str, qualities: list[str]):
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(24, 12))
    df_ecg = df[signal]
    ax.plot(df_ecg, label=f"Cushion Signal: {signal}")
    # Detect the peaks:
    peaks = ecg_r_peak_detection(df_segmented=[df], signal_name= "ecg1")[0]
    # Scatter the detected peaks: 
    ax.scatter(
        df_ecg.index[peaks],
        df_ecg.iloc[peaks],
        linewidth=2,
        color="red",
        alpha=1,
        label="Detected Peaks - Cushion Signal",
    )      
    ax.set_ylabel("Voltage [mV]")

    # Reference Signal:     
    plot_reference_signal(df_ecg=df_ecg, ax = ax)

    
    ax.grid()  
    ax.set_xlabel("Time [s]")
    fig.legend()
    signal_info = df.attrs["signal"]
    fig.suptitle(f"Participant: {signal_info.split('parti: ')[-1].split(' |')[0]} | Segment: {signal_info.split('segment: ')[-1]}\nSQ: {qualities[0]}, MA: {qualities[1]}" )
    return fig
    
def plot_single_data_with_hr_rr(df: pd.DataFrame, signal: str, qualities: list[str]):
    fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(24, 12))
    df_ecg = df[signal]
    ax[0].plot(df_ecg, label=f"Cushion Signal: {signal}")
    # Detect the peaks:
    peaks = ecg_r_peak_detection(df_segmented=[df], signal_name= "ecg1")[0]
    # Scatter the detected peaks: 
    ax[0].scatter(
        df_ecg.index[peaks],
        df_ecg.iloc[peaks],
        linewidth=2,
        color="red",
        alpha=1,
        label="Detected Peaks - Cushion Signal",
    )      
    ax[0].set_ylabel("Voltage [mV]")

    # Reference Signal:     
    plot_reference_signal(df_ecg=df_ecg, ax = ax[0])

    # Plot HR- Signal
    plot_hr_signal(df=df_ecg, peaks=peaks, ax=ax[1], label="HR - Cushion Signal")

    # Reference Signal:     
    df_ref, peaks_ref = plot_reference_signal(df_ecg=df_ecg, ax = ax[0], legend=True)

    # Plot HR- Ref
    plot_hr_signal(df=df_ref, peaks=peaks_ref, ax= ax[1], label= "HR- Reference Signal")

    
    ax[0].grid()
    ax[1].grid()
    ax[0].set_xlabel("Time [s]")
    ax[1].set_xlabel("Time [s]")
    ax[0].legend()
    ax[1].legend()
    signal_info = df.attrs["signal"]
    fig.suptitle(f"Participant: {signal_info.split('parti: ')[-1].split(' |')[0]} | Segment: {signal_info.split('segment: ')[-1]}\nSQ: {qualities[0]}, MA: {qualities[1]}" )
    return fig

In [10]:
signals = get_random_signals(("good", "no_ma"), ("good", "no_ma"), ("good", "no_ma"))
triple_fig = plot_triple_data(dfs = signals, signal=SIGNAL, title= [f"Good - {signals[0].attrs['signal']}" , f"Good - {signals[1].attrs['signal']}", f"Good - {signals[2].attrs['signal']}"])
# single_fig = plot_single_data(df = signals[0], signal=SIGNAL, qualities= ["Good", "No MA"])
# single_fig_with_hr_rr = plot_single_data_with_hr_rr(df = signals[0], signal=SIGNAL, qualities= ["Good", "No MA"])
# single_fig.savefig("test.pgf")


IndexError: list index out of range